# Zero to Chinese LLM · 125M demo

这是一个**从随机权重训练出来的 125M 中文 GPT**（不是 API，也不是蒸馏）：

- 语料：71,665 篇 / 112.5M tokens（中文维基百科 + 维基新闻 + 维基导游）
- 预训练：125.75M 参数，3 个 epoch，4090D 上 114.3k tok/s
- SFT：24,000 条 ChatML 样本，assistant-only loss，val_loss 3.807

⚠️ **预期管理**：它能把中文写通顺，但**事实性不可靠**，不能替代任何助手。
这是用来观察"训练链路产物"的，不是用来做题的。

点 ▶️ 从上往下跑即可。

In [ ]:
!pip -q install torch --index-url https://download.pytorch.org/whl/cpu
!pip -q install sentencepiece gradio

In [ ]:
import os, sys, subprocess, pathlib
if not pathlib.Path("zero-to-chinese-llm").exists():
    subprocess.run(["git", "clone", "--depth", "1",
                    "https://github.com/TanyaGaotiantian/zero-to-chinese-llm"], check=True)
os.chdir("zero-to-chinese-llm")
!wget -q -O gpt125m-sft-bf16.pt https://github.com/TanyaGaotiantian/zero-to-chinese-llm/releases/download/v1.0-models/gpt125m-sft-bf16.pt
!wget -q -O zh_bpe.model https://github.com/TanyaGaotiantian/zero-to-chinese-llm/releases/download/v1.0-models/zh_bpe.model
print("weights:", round(os.path.getsize("gpt125m-sft-bf16.pt")/2**20), "MiB")

In [ ]:
import sys, torch, sentencepiece as spm
sys.path.insert(0, "code/llm")
from model import GPT, GPTConfig

IM_START, IM_END = 32000, 32001
sp = spm.SentencePieceProcessor(model_file="zh_bpe.model")
ck = torch.load("gpt125m-sft-bf16.pt", map_location="cpu", weights_only=False)
cfg = GPTConfig(**{k: v for k, v in (ck.get("config") or {}).items() if k in GPTConfig.__annotations__})
device = "cuda" if torch.cuda.is_available() else "cpu"
model = GPT(cfg).to(device).eval()
model.load_state_dict(ck["model"], strict=False)
print(f"{sum(p.numel() for p in model.parameters())/1e6:.2f}M params on {device}, step={ck.get('step')}")

In [ ]:
import torch.nn.functional as F

def build_prompt(messages):
    ids = []
    for m in messages:
        ids.append(IM_START)
        ids += sp.encode(f"{m['role']}\n", out_type=int)
        ids += sp.encode(m["content"], out_type=int)
        ids.append(IM_END)
    ids.append(IM_START)
    ids += sp.encode("assistant\n", out_type=int)
    return ids

@torch.no_grad()
def generate(messages, max_new_tokens=120, temperature=0.7, top_p=0.9, rep=1.3, window=64):
    ids = build_prompt(messages)
    idx = torch.tensor([ids[-cfg.seq_len:]], device=device)
    recent = list(idx[0].tolist())[-window:]
    out = []
    for _ in range(max_new_tokens):
        logits, _ = model(idx[:, -cfg.seq_len:])
        logits = logits[:, -1, :].float()
        if recent:
            uniq = torch.tensor(sorted(set(recent)), device=device)
            v = logits[0, uniq]
            logits[0, uniq] = torch.where(v > 0, v / rep, v * rep)
        logits = logits / max(temperature, 1e-4)
        srt, idxs = torch.sort(logits, descending=True, dim=-1)
        probs = F.softmax(srt, dim=-1).cumsum(dim=-1)
        cut = probs > top_p
        cut[..., 1:] = cut[..., :-1].clone(); cut[..., 0] = False
        srt[cut] = -float("inf")
        logits = torch.full_like(logits, -float("inf")).scatter(1, idxs, srt)
        nid = int(torch.multinomial(F.softmax(logits, dim=-1), num_samples=1))
        if nid == IM_END:
            break
        out.append(nid)
        recent = (recent + [nid])[-window:]
        idx = torch.cat([idx, torch.tensor([[nid]], device=device)], dim=1)
    return sp.decode(out).strip()

for q in ["用一句话解释什么是机器学习", "列出三个学习编程的建议", "给一家咖啡店起三个名字"]:
    print("你>", q)
    print("模型>", generate([{"role": "user", "content": q}]), "\n" + "-" * 60)

In [ ]:
# 想要一个可点击的界面：跑这一格（Colab 会给出一个临时公网链接）
import gradio as gr

def respond(message, history, temperature, top_p, rep, max_new):
    history = history or []
    if not message.strip():
        return history, ""
    msgs = []
    for u, a in history:
        if u: msgs.append({"role": "user", "content": u})
        if a: msgs.append({"role": "assistant", "content": a})
    msgs.append({"role": "user", "content": message})
    reply = generate(msgs, max_new_tokens=int(max_new), temperature=temperature, top_p=top_p, rep=rep)
    return history + [[message, reply or "(空)"]], ""

with gr.Blocks(title="Zero to Chinese LLM · 125M") as demo:
    gr.Markdown("# 125M 中文 GPT demo\n只用 112.5M tokens 从零训练，能把中文写通顺，但事实性不可靠。")
    chat = gr.Chatbot(height=340)
    with gr.Row():
        box = gr.Textbox(placeholder="问点什么……", scale=4)
        btn = gr.Button("发送", variant="primary", scale=1)
    with gr.Row():
        t = gr.Slider(0.1, 1.5, value=0.7, step=0.05, label="temperature")
        p = gr.Slider(0.3, 1.0, value=0.9, step=0.05, label="top-p")
        r = gr.Slider(1.0, 1.6, value=1.3, step=0.05, label="repetition penalty")
        m = gr.Slider(32, 300, value=120, step=8, label="max new tokens")
    btn.click(respond, [box, chat, t, p, r, m], [chat, box])
    box.submit(respond, [box, chat, t, p, r, m], [chat, box])

demo.launch(share=True, debug=False)